In [ ]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import numpy as np

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")
ussmallcap = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CUSS ETF Stock Price History.csv")
silver = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SSLN ETF Stock Price History.csv")
energy = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "MLPX ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope, ussmallcap, silver, energy]
labels = ["SNP", "China", "EM", "Gold", "India", "MSCI-Europe", "SmallCapEurope", "USSmallCap", "Silver", "Energy"]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)
# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]
print([len(x) for x in dfs])

N = len(dfs)
T = len(dfs[0]) - 1

dfs = [df[::-1] for df in dfs]

returns_all = np.empty((T, N))
for i, df in enumerate(dfs):
    returns_all[:, i] = df["Price"].pct_change().values[1:]

corr = np.corrcoef(returns_all, rowvar=False)

heatmap = go.Figure()
heatmap.add_trace(go.Heatmap(x=labels, y=labels, z=corr))
heatmap.show()

[2362, 2362, 2362, 2362, 2362, 2362, 2362, 2362, 2362, 2362]
            Date   Price    Open    High     Low     Vol. Change %  Vol_clean
2736  2015-06-04  193.48  194.56  194.59  193.39   79.92K   -0.76%    79920.0
2735  2015-06-05  192.96  193.31  193.56  192.13  268.78K   -0.27%   268780.0
2734  2015-06-08  192.07  192.74  192.74  192.04   96.24K   -0.46%    96240.0
2733  2015-06-09  191.68  191.65  191.70  190.64  114.16K   -0.20%   114160.0
2732  2015-06-10  193.84  191.76  194.00  191.50  150.58K    1.13%   150580.0
...          ...     ...     ...     ...     ...      ...      ...        ...
4     2026-03-16  717.87  715.53  721.96  713.82   67.65K    0.50%    67650.0
3     2026-03-17  721.96  716.56  725.07  715.95   80.25K    0.57%    80250.0
2     2026-03-18  716.48  724.70  725.53  715.85  132.07K   -0.76%   132070.0
1     2026-03-19  707.36  709.91  711.88  704.51  123.68K   -1.27%   123680.0
0     2026-03-20  703.62  710.74  711.37  702.18  195.33K   -0.53%   195330.0

[2

In [12]:
# estimate sharp using equal weights
N = len(dfs)
w = np.ones(N) / N
rf = 0.0
cov = np.cov(returns_all, rowvar=False)
mu = np.mean(returns_all, axis=0)
ret = np.dot(mu, w)
vola = np.sqrt(w.T @ cov @ w)
print(ret / vola * np.sqrt(255))

0.7695654138453178


In [15]:
N_prime = N - 1
considering = returns_all[:, :-1]
w = np.ones(N_prime) / N_prime
rf = 0.0
cov = np.cov(considering, rowvar=False)
mu = np.mean(considering, axis=0)
ret = np.dot(mu, w)
vola = np.sqrt(w.T @ cov @ w)
print(ret / vola * np.sqrt(255))

0.8092532716133471


In [18]:
graph = go.Figure()
for i in range(returns_all.shape[1]):
    L = list(range(T))
    cum_returns = np.cumprod(1 + returns_all[:,i]) - 1
    graph.add_trace(go.Scatter(x=L, y=cum_returns, name=labels[i]))
graph.show()